In [1]:
import numpy as np
import torch
import spacy
import torch.nn as nn
from torch.utils.data import DataLoader
from torch.nn.utils.rnn import pad_sequence
from torchtext.vocab import build_vocab_from_iterator
import datasets
import tqdm

/Users/vishwa/Projects/LanguageTranslator.ai/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
# !python3 -m spacy download de_core_news_sm
# !python3 -m spacy download en_core_web_sm

In [2]:
language_dataset = datasets.load_dataset('bentrevett/multi30k')
train_data,val_data,test_data = language_dataset['train'],language_dataset['validation'],language_dataset['test']

In [ ]:
train_length = len(train_data)
val_length = len(val_data)
test_length = len(test_data)

print(f"Train data len: {train_length}")
print(f"Validation data len: {val_length}")
print(f"Test data len: {test_length}")

Train data leng: 29000
Validation data len: 1014
Test data leng: 1000


In [6]:
train_data[:5]

{'en': ['Two young, White males are outside near many bushes.',
  'Several men in hard hats are operating a giant pulley system.',
  'A little girl climbing into a wooden playhouse.',
  'A man in a blue shirt is standing on a ladder cleaning a window.',
  'Two men are at the stove preparing food.'],
 'de': ['Zwei junge weiße Männer sind im Freien in der Nähe vieler Büsche.',
  'Mehrere Männer mit Schutzhelmen bedienen ein Antriebsradsystem.',
  'Ein kleines Mädchen klettert in ein Spielhaus aus Holz.',
  'Ein Mann in einem blauen Hemd steht auf einer Leiter und putzt ein Fenster.',
  'Zwei Männer stehen am Herd und bereiten Essen zu.']}

In [7]:
en_nlp = spacy.load('en_core_web_sm'); de_nlp = spacy.load('de_core_news_sm')

In [8]:
[token.text for token in en_nlp.tokenizer("Two men are at the stove preparing food.")][:1000]

['Two', 'men', 'are', 'at', 'the', 'stove', 'preparing', 'food', '.']

In [ ]:
def tokenizer(sample,en_nlp,de_nlp,max_length,sos_token,eos_token):
    en_tokens = [token.text for token in en_nlp.tokenizer(sample["en"])][:max_length]
    de_tokens = [token.text for token in de_nlp.tokenizer(sample["de"])][:max_length]
    en_tokens = [token.lower() for token in en_tokens]
    de_tokens = [token.lower() for token in de_tokens]
    en_tokens = [sos_token] + en_tokens + [eos_token]
    de_tokens = [sos_token] + de_tokens + [eos_token]
    # print(en_tokens, de_tokens)
    return {"en_tokens":en_tokens,"de_tokens":de_tokens}

In [ ]:
fn_kwargs = {"en_nlp":en_nlp, "de_nlp":de_nlp, "max_length":1000, "sos_token":'<sos>', "eos_token":'<eos>'}
train_data = train_data.map(tokenizer,fn_kwargs=fn_kwargs)
val_data = val_data.map(tokenizer,fn_kwargs=fn_kwargs)
test_data = test_data.map(tokenizer,fn_kwargs=fn_kwargs)

In [11]:
train_data[:1]

{'en': ['Two young, White males are outside near many bushes.'],
 'de': ['Zwei junge weiße Männer sind im Freien in der Nähe vieler Büsche.'],
 'en_tokens': [['<sos>',
   'two',
   'young',
   ',',
   'white',
   'males',
   'are',
   'outside',
   'near',
   'many',
   'bushes',
   '.',
   '<eos>']],
 'de_tokens': [['<sos>',
   'zwei',
   'junge',
   'weiße',
   'männer',
   'sind',
   'im',
   'freien',
   'in',
   'der',
   'nähe',
   'vieler',
   'büsche',
   '.',
   '<eos>']]}

In [12]:
specials = ['<unk>','<pad>','<sos>','<eos>']
en_vocab = build_vocab_from_iterator(train_data['en_tokens'],specials=specials)
de_vocab = build_vocab_from_iterator(train_data['de_tokens'],specials=specials)

unk_index = en_vocab['<unk>']
pad_index = en_vocab['<pad>']
en_vocab.set_default_index(unk_index)
de_vocab.set_default_index(unk_index)

In [13]:
print("En vocab:")
for i, word in enumerate(en_vocab.get_itos()):  
    if i == 10:
        break
    print(f"Index: {i}, Word: {word}")

print("\nDe vocab:")
for i, word in enumerate(de_vocab.get_itos()):  
    if i == 10:
        break
    print(f"Index: {i}, Word: {word}")

En vocab:
Index: 0, Word: <unk>
Index: 1, Word: <pad>
Index: 2, Word: <sos>
Index: 3, Word: <eos>
Index: 4, Word: a
Index: 5, Word: .
Index: 6, Word: in
Index: 7, Word: the
Index: 8, Word: on
Index: 9, Word: man

De vocab:
Index: 0, Word: <unk>
Index: 1, Word: <pad>
Index: 2, Word: <sos>
Index: 3, Word: <eos>
Index: 4, Word: .
Index: 5, Word: ein
Index: 6, Word: einem
Index: 7, Word: in
Index: 8, Word: eine
Index: 9, Word: ,


In [14]:
print(f"En vocab len: {len(en_vocab)}")
print(f"De vocab len: {len(de_vocab)}")


En vocab len: 9797
De vocab len: 18669


In [15]:
def numericalize(sample,en_vocab,de_vocab):
    en_ids = en_vocab.lookup_indices(sample["en_tokens"])
    de_ids = de_vocab.lookup_indices(sample["de_tokens"])
    return {"en_ids":en_ids,"de_ids":de_ids}
fn_kwargs = {"en_vocab":en_vocab,"de_vocab":de_vocab}
train_data = train_data.map(numericalize,fn_kwargs=fn_kwargs)
val_data = val_data.map(numericalize,fn_kwargs=fn_kwargs)
test_data = test_data.map(numericalize,fn_kwargs=fn_kwargs)

Map: 100%|██████████| 1000/1000 [00:00<00:00, 20588.88 examples/s]


In [16]:
train_data[:1]

{'en': ['Two young, White males are outside near many bushes.'],
 'de': ['Zwei junge weiße Männer sind im Freien in der Nähe vieler Büsche.'],
 'en_tokens': [['<sos>',
   'two',
   'young',
   ',',
   'white',
   'males',
   'are',
   'outside',
   'near',
   'many',
   'bushes',
   '.',
   '<eos>']],
 'de_tokens': [['<sos>',
   'zwei',
   'junge',
   'weiße',
   'männer',
   'sind',
   'im',
   'freien',
   'in',
   'der',
   'nähe',
   'vieler',
   'büsche',
   '.',
   '<eos>']],
 'en_ids': [[2, 16, 24, 15, 25, 778, 17, 57, 80, 202, 1312, 5, 3]],
 'de_ids': [[2, 18, 26, 253, 30, 84, 20, 88, 7, 15, 110, 7647, 3171, 4, 3]]}

In [17]:
train_data = train_data.with_format(type="torch",columns=['en_ids','de_ids'],output_all_columns=True)
val_data = val_data.with_format(type="torch",columns=['en_ids','de_ids'],output_all_columns=True)
test_data = test_data.with_format(type="torch",columns=['en_ids','de_ids'],output_all_columns=True)

In [18]:
train_data[:1]

{'en_ids': tensor([[   2,   16,   24,   15,   25,  778,   17,   57,   80,  202, 1312,    5,
             3]]),
 'de_ids': tensor([[   2,   18,   26,  253,   30,   84,   20,   88,    7,   15,  110, 7647,
          3171,    4,    3]]),
 'en': ['Two young, White males are outside near many bushes.'],
 'de': ['Zwei junge weiße Männer sind im Freien in der Nähe vieler Büsche.'],
 'en_tokens': [['<sos>',
   'two',
   'young',
   ',',
   'white',
   'males',
   'are',
   'outside',
   'near',
   'many',
   'bushes',
   '.',
   '<eos>']],
 'de_tokens': [['<sos>',
   'zwei',
   'junge',
   'weiße',
   'männer',
   'sind',
   'im',
   'freien',
   'in',
   'der',
   'nähe',
   'vieler',
   'büsche',
   '.',
   '<eos>']]}

In [ ]:
def get_collate_fn(pad_index):
    def collate_fn(batch):
        batch_en_ids = [sample["en_ids"] for sample in batch]
        batch_de_ids = [sample["de_ids"] for sample in batch]
        batch_en_ids = pad_sequence(batch_en_ids,padding_value=pad_index)
        batch_de_ids = pad_sequence(batch_de_ids,padding_value=pad_index)
        batch = {"en_ids":batch_en_ids,"de_ids":batch_de_ids}
        return batch
    return collate_fn
def dataloader_func(dataset,batch_size,shuffle,pad_index):
    collate_fn = get_collate_fn(pad_index)
    dataloader = DataLoader(dataset=dataset,batch_size=batch_size,shuffle=shuffle,collate_fn=collate_fn)
    return dataloader

In [20]:
train_loader = dataloader_func(train_data,batch_size=512,shuffle=True,pad_index=pad_index)
val_loader = dataloader_func(val_data,batch_size=512,shuffle=True,pad_index=pad_index)
test_loader = dataloader_func(test_data,batch_size=512,shuffle=True,pad_index=pad_index)

In [21]:
class Encoder(nn.Module):
    def __init__(self,input_dim,embedding_dim,hidden_size,num_layers,dropout):
        super(Encoder,self).__init__()
        self.hidden_size = hidden_size
        self.num_layers = num_layers
        self.dropout = nn.Dropout(dropout)
        self.embedding = nn.Embedding(input_dim,embedding_dim)
        self.lstm = nn.LSTM(embedding_dim,hidden_size,num_layers=num_layers,bidirectional=True,dropout=dropout)
    def forward(self,src):
        embedded = self.dropout(self.embedding(src))
        out,(hidden,cell) = self.lstm(embedded)
        return hidden,cell

In [22]:
class Decoder(nn.Module):
    def __init__(self,output_dim,embedding_dim,hidden_size,num_layers,dropout):
        super(Decoder,self).__init__()
        self.output_dim = output_dim
        self.hidden_size = hidden_size
        self.num_layers = num_layers
        self.dropout = nn.Dropout(dropout)
        self.embedding = nn.Embedding(output_dim,embedding_dim)
        self.lstm = nn.LSTM(embedding_dim,hidden_size,num_layers=num_layers,bidirectional=True,dropout=dropout)
        self.fc = nn.Linear(hidden_size*2,output_dim)
    def forward(self,input_token,hidden,cell):
        input_token = input_token.unsqueeze(0)
        emb = self.embedding(input_token)
        emb = self.dropout(emb)
        out,(hidden,cell) = self.lstm(emb,(hidden,cell))
        out = out.squeeze(0)
        pred = self.fc(out)
        return pred,hidden,cell

In [23]:
class Seq2Seq(nn.Module):
    def __init__(self,encoder,decoder,device):
        super(Seq2Seq,self).__init__()
        self.encoder = encoder
        self.decoder = decoder
        self.device = device
    def forward(self,src,trg,teacher_forcing_ratio):
        trg_len = trg.shape[0]
        batch_size = trg.shape[1]
        vocab_size = self.decoder.output_dim
        outputs = torch.zeros(trg_len,batch_size,vocab_size).to(self.device)
        input_token = trg[0,:]
        hidden,cell = self.encoder(src)
        for t in range(1,trg_len):
            out,hidden,cell = self.decoder(input_token,hidden,cell)
            outputs[t] = out
            top1 = out.argmax(1)
            teacher_force = np.random.randn()<teacher_forcing_ratio
            input_token = trg[t] if teacher_force else top1
        return outputs

In [24]:
input_dim = len(de_vocab)
output_dim = len(en_vocab)
encoder_embedding_dim = 256
decoder_embedding_dim = 256
hidden_size = 512
num_layers = 3
encoder_dropout = 0.2
decoder_dropout = 0.2
device = torch.device("mps" if torch.cuda.is_available() else "cpu")

encoder = Encoder(input_dim=input_dim, embedding_dim=encoder_embedding_dim, hidden_size=hidden_size, num_layers=num_layers, dropout=encoder_dropout,)

decoder = Decoder(output_dim=output_dim, embedding_dim=decoder_embedding_dim, hidden_size=hidden_size, num_layers=num_layers, dropout=decoder_dropout,)

model = Seq2Seq(encoder, decoder, device).to(device)

In [25]:
optimizer = torch.optim.Adam(model.parameters())
criterion = nn.CrossEntropyLoss(ignore_index=pad_index)

In [26]:
def train_fn( model, data_loader, optimizer, criterion, clip, teacher_forcing_ratio, device):
    model.train()
    epoch_loss = 0
    for i, batch in enumerate(data_loader):
        src = batch["de_ids"].to(device)
        trg = batch["en_ids"].to(device)
        optimizer.zero_grad()
        output = model(src, trg, teacher_forcing_ratio)
        output_dim = output.shape[-1]
        output = output[1:].view(-1, output_dim)
        trg = trg[1:].view(-1)
        loss = criterion(output, trg)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), clip)
        optimizer.step()
        epoch_loss += loss.item()
    return epoch_loss / len(data_loader)


In [ ]:
def evaluate_fn(model, data_loader, criterion, device):
    model.eval()
    epoch_loss = 0
    correct_predictions = 0  
    total_predictions = 0

    with torch.no_grad():
        for i, batch in enumerate(data_loader):
            src = batch["de_ids"].to(device)
            trg = batch["en_ids"].to(device)
            output = model(src, trg, 0)
            output_dim = output.shape[-1]
            output = output[1:].view(-1, output_dim)
            trg = trg[1:].view(-1)

            loss = criterion(output, trg)
            epoch_loss += loss.item()

            pred = output.argmax(dim=-1)

            non_pad_mask = trg.ne(pad_index)
            correct = pred.eq(trg).masked_select(non_pad_mask).sum().item()
            correct_predictions += correct
            total_predictions += non_pad_mask.sum().item()
    accuracy = correct_predictions / total_predictions * 100
    avg_loss = epoch_loss / len(data_loader)
    
    return avg_loss, accuracy


In [ ]:
n_epochs = 5
clip = 1.0
teacher_forcing_ratio = 1

best_valid_loss = float("inf")

for epoch in tqdm.tqdm(range(n_epochs)):
    train_loss = train_fn(model, train_loader, optimizer, criterion, clip,
teacher_forcing_ratio, device,)
    valid_loss = evaluate_fn(model, val_loader, criterion, device,)
    if valid_loss < best_valid_loss:
        best_valid_loss = valid_loss
        torch.save(model.state_dict(), "seq2seq.pt")
    print(f"\tTraining Loss: {train_loss:7.3f} | Training Perplexity score: {np.exp(train_loss):7.3f}")
    print(f"\tValiding Loss: {valid_loss:7.3f} | Validing Perplexity score: {np.exp(valid_loss):7.3f}")

 20%|██        | 1/5 [24:46<1:39:04, 1486.13s/it]

	Training Loss:   4.555 | Training Perplexity score:  95.143
	Validing Loss:   4.428 | Validing Perplexity score:  83.742


 40%|████      | 2/5 [58:50<1:30:43, 1814.41s/it]

	Training Loss:   4.066 | Training Perplexity score:  58.333
	Validing Loss:   4.148 | Validing Perplexity score:  63.281


 60%|██████    | 3/5 [1:20:57<53:04, 1592.07s/it]

	Training Loss:   3.752 | Training Perplexity score:  42.610
	Validing Loss:   3.986 | Validing Perplexity score:  53.813


 80%|████████  | 4/5 [1:42:01<24:22, 1462.49s/it]

	Training Loss:   3.530 | Training Perplexity score:  34.111
	Validing Loss:   3.897 | Validing Perplexity score:  49.269


100%|██████████| 5/5 [2:05:37<00:00, 1507.53s/it]

	Training Loss:   3.334 | Training Perplexity score:  28.051
	Validing Loss:   3.587 | Validing Perplexity score:  36.126


In [32]:
model.load_state_dict(torch.load("models/seq2seq.pt"))

test_loss, test_accuracy = evaluate_fn(model, test_loader, criterion, device)

print(f"Test Loss: {test_loss:.3f} | Test Accuracy: {test_accuracy:.2f}% | Test Perplexity: {np.exp(test_loss):7.3f}")

Test Loss: 3.516 | Test Accuracy: 41.83% | Test Perplexity:  33.637


In [33]:
def translate_sentence(sentence, model, en_nlp, de_nlp, en_vocab, de_vocab, sos_token, eos_token, device, max_output_length=25,):
    model.eval()
    with torch.no_grad():
        if isinstance(sentence, str):
            tokens = [token.text for token in de_nlp.tokenizer(sentence)]
        else:
            tokens = [token for token in sentence]
        
        tokens = [token.lower() for token in tokens]
        tokens = [sos_token] + tokens + [eos_token]
        ids = de_vocab.lookup_indices(tokens)
        tensor = torch.LongTensor(ids).unsqueeze(-1).to(device)
        hidden, cell = model.encoder(tensor)
        inputs = en_vocab.lookup_indices([sos_token])
        for _ in range(max_output_length):
            inputs_tensor = torch.LongTensor([inputs[-1]]).to(device)
            output, hidden, cell = model.decoder(inputs_tensor, hidden, cell)
            predicted_token = output.argmax(-1).item()
            inputs.append(predicted_token)
            if predicted_token == en_vocab[eos_token]:
                break
        tokens = en_vocab.lookup_tokens(inputs)
    return tokens

In [36]:
sentence ='Mehrere Männer mit Schutzhelmen bedienen ein Antriebsradsystem'
#Several men in hard hats are operating a giant pulley system.
sos_token='<sos>'
eos_token='<eos>'
translation = translate_sentence(sentence,model,en_nlp,de_nlp,en_vocab,de_vocab,sos_token,eos_token,device,)
print(translation)

['<sos>', 'several', 'men', 'in', 'a', 'room', ',', 'a', 'a', 'man', '.', '<eos>']
